# Exploring explainkit: bike rentals

We train a model that forecasts hourly bike rentals (Washington DC, public UCI dataset), attach **explainkit**
to it, and ask it questions. Every answer prints in **plain English**, built only from exact computed numbers (no LLM needed).
The numbers are still there underneath: `r["change"]`, `dict(r)`.

Run cells top to bottom with **Shift+Enter**. Change the inputs and rerun to explore.

## Step 1: import the library

In [1]:
import json
import lightgbm as lgb
import pandas as pd
from sklearn.datasets import fetch_openml

from explainkit import Explainer   # <- the library

show = lambda r: print(json.dumps(r, indent=2))

## Step 2: load data and train a model
Any model works. Here: LightGBM, trained on 2011 and used to forecast 2012.

In [2]:
df = fetch_openml(data_id=42712, as_frame=True, data_home="../data").frame
features = [c for c in df.columns if c != "count"]

train = df[df["year"] == 0]                       # 2011
test = df[df["year"] == 1].reset_index(drop=True)  # 2012

model = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.05, random_state=0, verbose=-1)
model.fit(train[features], train["count"])
test.head()

,season,year,month,hour,holiday,weekday,workingday,weather,temp,feel_temp,humidity,windspeed,count
0,spring,1,1,0,False,0,False,clear,14.76,18.940,0.66,0.0000,48
1,spring,1,1,1,False,0,False,clear,14.76,17.425,0.66,8.9981,93
2,spring,1,1,2,False,0,False,clear,13.12,17.425,0.76,0.0000,75
3,spring,1,1,3,False,0,False,clear,12.30,16.665,0.81,0.0000,52
4,spring,1,1,4,False,0,False,clear,11.48,15.150,0.81,6.0032,8


## Step 3: attach explainkit to the model
- `target_name`: what the model predicts, in words (used in answers)
- `y`: the actual outcomes, so it can answer accuracy questions
- `feature_groups`: report related inputs as one factor
- `llm`: which AI writes the plain-English answers (only used by `ex.ask`)
- `value_labels`: show codes as words (weekday 2 → Tuesday, holiday True → holiday)

In [3]:
ex = Explainer(
    model, test, target_name="bike rentals per hour", y="count",
    feature_groups={"weather conditions": ["weather", "temp", "feel_temp", "humidity", "windspeed"],
                    "day type": ["holiday", "workingday", "weekday"],
                    "time of day": ["hour"], "time of year": ["season", "month"]},
    feature_descriptions={"year": "0 = 2011, 1 = 2012", "weekday": "0 = Sunday ... 6 = Saturday",
                          "feel_temp": "apparent temperature, Celsius", "humidity": "0 to 1"},
    value_labels={
        "weekday": dict(enumerate(["Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"])),
        "month": dict(enumerate(["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"], 1)),
        "hour": lambda h: f"{h}:00",
        "holiday": {"True": "holiday", "False": "not a holiday"},
        "workingday": {"True": "working day", "False": "day off"},
        "temp": lambda t: f"{t:.0f}°C", "feel_temp": lambda t: f"feels like {t:.0f}°C",
        "humidity": lambda h: f"{h:.0%} humidity", "windspeed": lambda w: f"wind {w:.0f} km/h",
    },
    llm="gemini/gemini-2.5-flash",
)

## The two forecasts we compare
Same hour (Friday 5pm), one week apart. `count` is what actually happened.

In [4]:
cols = ["month", "weekday", "hour", "weather", "temp", "humidity", "windspeed", "workingday", "count"]
friday_5pm = test[(test["weekday"] == 5) & (test["hour"] == 17)]
last_week, this_week = friday_5pm.iloc[[20]], friday_5pm.iloc[[21]]
pd.concat([last_week[cols], this_week[cols]], keys=["last week", "this week"])

,,month,weekday,hour,weather,temp,humidity,windspeed,workingday,count
last week,3487,5,5,17,clear,31.16,0.62,19.0012,True,653
this week,3655,6,5,17,rain,28.70,0.70,16.9979,True,227


## 1. Why is this forecast what it is?
Start from the average hour, add each factor's effect, and you land exactly on the forecast.

In [5]:
print(ex.explain_prediction(this_week))

The model predicts 525 bike rentals per hour, 381 above the average of 144.
Why:
  • time of day (17:00): +251
  • weather conditions (rain, 29°C, feels like 33°C, ...): +83.6
      (weather −22.4 pulled it down, but temp +51.8, feel_temp +39.6 pushed it up more)
  • time of year (summer, Jun): +30.3
      (made up of: month +23.8, season +6.6)
  • day type (not a holiday, working day, Friday): +15.9
      (weekday −4.7 pulled it down, but workingday +20.4 pushed it up more)


## 2. Is that high or low?

In [6]:
print(ex.prediction_context(this_week))

525 bike rentals per hour is unusually high: higher than 98% of all predictions. Most predictions fall between 9.3 and 352 (average 149).


## 3. Why did it change since last week?
Only inputs that actually changed get credit. Inputs that stayed the same get exactly zero.

In [7]:
print(ex.explain_change(before=last_week, after=this_week))

The prediction went from 529 to 525 bike rentals per hour: down by 4 (−0.76%).
What caused it:
  • weather conditions (clear → rain, 31°C → 29°C, feels like 36°C → feels like 33°C, ...): −5.2
      (feel_temp +16, windspeed +7.1 pushed it up, but humidity −16.5, weather −10.9 pulled it down more)
  • time of year (May → Jun): +1.1, 28% of the change
Inputs that didn't change contributed nothing.


### Under the hood: the exact numbers
Every answer is also a dict of exact numbers. This is what an LLM or your own code reads.

In [8]:
r = ex.explain_change(last_week, this_week)
print(r["before"], "→", r["after"], "change:", r["change"])
show(r)

528.6 → 524.6 change: -4.039
{
  "target": "bike rentals per hour",
  "before": 528.6,
  "after": 524.6,
  "change": -4.039,
  "pct_change": -0.7641,
  "drivers": [
    {
      "factor": "weather conditions",
      "effect": -5.107,
      "before": {
        "weather": "clear",
        "temp": "31\u00b0C",
        "feel_temp": "feels like 36\u00b0C",
        "humidity": "62% humidity",
        "windspeed": "wind 19 km/h"
      },
      "after": {
        "weather": "rain",
        "temp": "29\u00b0C",
        "feel_temp": "feels like 33\u00b0C",
        "humidity": "70% humidity",
        "windspeed": "wind 17 km/h"
      },
      "share_of_change_pct": 126.4,
      "parts": {
        "humidity": -16.59,
        "feel_temp": 16.07,
        "weather": -10.81,
        "windspeed": 7.157,
        "temp": -0.9287
      }
    },
    {
      "factor": "time of year",
      "effect": 1.068,
      "before": {
        "season": "summer",
        "month": "May"
      },
      "after": {
        

## 4. What if...?
Change any input. Operations: `set`, `add`, `multiply`. Try `weather` set to `clear`.

In [9]:
print(ex.what_if(this_week, [{"feature": "weather", "operation": "set", "value": "clear"}]))

Changing rain → clear moves the prediction from 525 to 535 bike rentals per hour: up by 10.5 (+2%).
This is how the model responds, not proof the change would cause this in reality.


## 5. What matters most? Overall, then for a segment

In [10]:
print(ex.top_drivers())

What matters most for bike rentals per hour (all data, 8,734 rows):
  1. time of day: typically moves a prediction by 82.8
  2. weather conditions: typically moves a prediction by 34.7
  3. time of year: typically moves a prediction by 20.8
  4. day type: typically moves a prediction by 18.4


In [11]:
print(ex.top_drivers({"workingday": "False"}))   # weekends and holidays

What matters most for bike rentals per hour (workingday = False, 2,780 rows):
  1. time of day: typically moves a prediction by 74.7
  2. weather conditions: typically moves a prediction by 39.3
  3. day type: typically moves a prediction by 25.6
  4. time of year: typically moves a prediction by 19.3
Average prediction here: 145 vs 149 overall.


## 6. How accurate is the model, and where is it worst?
`bias` < 0 means it under-predicts on average.

In [12]:
print(ex.model_errors(by="season"))

On all data (8,734 rows), the model is typically off by 89.5 bike rentals per hour, 38.1% of the average actual (235).
It tends to under-predict, by 85.2 on average.
Worst season: fall (off by 98). Best: spring (off by 74.5).


## 7. Pick your own hour
Change the filter to any day and hour in 2012.

In [13]:
my_hour = test[(test["month"] == 7) & (test["weekday"] == 0) & (test["hour"] == 14)].iloc[[0]]
print(my_hour[cols].to_string())
print(ex.explain_prediction(my_hour))

      month  weekday  hour weather   temp  humidity  windspeed workingday  count
4372      7        0    14   clear  37.72      0.33    16.9979      False    444


The model predicts 274 bike rentals per hour, 131 above the average of 144.
Why:
  • time of day (14:00): +80.7
  • day type (not a holiday, day off, Sunday): +49.5
  • time of year (fall, Jul): +10.8
      (made up of: month +5.7, season +5.1)
  • weather conditions (clear, 38°C, feels like 42°C, ...): −10.4
      (humidity +11.5, weather +4.4 pushed it up, but feel_temp −24.5 pulled it down more)


## 8. In plain English (needs an LLM key)
Put `GEMINI_API_KEY=...` in `.env`, then uncomment and run:

In [14]:
# from dotenv import load_dotenv; load_dotenv()
# print(ex.ask("Why did the forecast change since last week?", row=this_week, compare_to=last_week))